In [ ]:
# pip install pycocoevalcap torchmetrics transformers
import sys
import json
import pandas as pd
import torch
from pathlib import Path

# Import các metrics từ pycocoevalcap
from pycocoevalcap.bleu.bleu import Bleu
from pycocoevalcap.meteor.meteor import Meteor
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.spice.spice import Spice

def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "src" / "config" / "common_config.py").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the project root")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config.clipcap_config import TOKENIZED_DIR, CLIPCAP_OUTPUT_ROOT, CLIP_MODEL_NAME

PREDICTIONS_PATH = CLIPCAP_OUTPUT_ROOT / "test_predictions.json"
TEST_DATA_PATH = TOKENIZED_DIR / "test.pt"

# Load và chuyển đổi dữ liệu sang Format của COCO

In [ ]:
# Cell 2: Hàm Format Data
def load_coco_format(gt_path: Path, pred_path: Path):
    """
    Load và chuyển đổi dữ liệu sang chuẩn MS COCO Eval.
    gts = { "image_id": [{"caption": "text1"}, {"caption": "text2"}] }
    res = { "image_id": [{"caption": "predicted text"}] }
    """
    if not gt_path.exists() or not pred_path.exists():
        raise FileNotFoundError("Không tìm thấy file Test hoặc Predictions!")
        
    test_data = torch.load(gt_path, map_location="cpu")
    with open(pred_path, 'r', encoding='utf-8') as f:
        preds = json.load(f)
        
    gts = {}
    res = {}
    
    # 1. Build Ground Truth dictionary
    for item in test_data:
        img_id = str(item['image_id'])
        if img_id not in gts:
            gts[img_id] = []
        gts[img_id].append({'caption': str(item['caption'])})
        
    # 2. Build Predictions dictionary
    for item in preds:
        img_id = str(item['image_id'])
        # Đảm bảo chỉ đánh giá trên các ảnh tồn tại trong tập Ground Truth
        if img_id in gts:
            res[img_id] = [{'caption': str(item['prediction'])}]
            
    return gts, res

print("Sẵn sàng format dữ liệu.")

# Chạy toán bộ NLP Metrics (CIDEr, METEOR, ROUGE-L, SPICE, BLEU)

In [ ]:
# Cell 3: Thực thi NLP Evaluation
def evaluate_all_nlp_metrics(gts, res):
    
    scorers = [
        (Bleu(4), ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4"]),
        (Meteor(), "METEOR"),
        (Rouge(), "ROUGE-L"),
        (Cider(), "CIDEr"),
        (Spice(), "SPICE")
    ]
    
    results_dict = {}
    
    for scorer, method in scorers:
        print(f"Đang tính toán {method}...")
        score, scores = scorer.compute_score(gts, res)
        
        if isinstance(method, list):
            for m, s in zip(method, score):
                results_dict[m] = round(s * 100, 2)
        else:
            results_dict[method] = round(score * 100, 2)
            
    return results_dict

try:
    gts, res = load_coco_format(TEST_DATA_PATH, PREDICTIONS_PATH)
    nlp_results = evaluate_all_nlp_metrics(gts, res)
    
    print("\n" + "="*35)
    print(" KẾT QUẢ ĐÁNH GIÁ NGỮ NGHĨA (NLP)")
    print("="*35)
    for metric, score in nlp_results.items():
        print(f" {metric:10}: {score}")
    print("="*35)
    
except Exception as e:
    print(f"Lỗi: {e}")

# Tính toán CLIPScore (Image-Text Similarity)

In [ ]:
# CLIPScore Evaluation (Reference-free metric)
from torchmetrics.multimodal.clip_score import CLIPScore
from PIL import Image

def calculate_clipscore(res_dict, image_folder_path: Path):
    """
    Tính CLIPScore yêu cầu folder chứa raw images của tập test.
    Thay 'image_folder_path' bằng đường dẫn tới folder ảnh thực tế.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Đang khởi tạo CLIPScore trên thiết bị: {device}")
    
    # Load model openai/clip-vit-base-patch32 từ cấu hình
    clip_scorer = CLIPScore(model_name_or_path=CLIP_MODEL_NAME).to(device)
    
    total_score = 0
    valid_images = 0
    
    print("Đang đối chiếu ảnh và predictions...")
    for img_id, pred_list in res_dict.items():
        # Giả sử ảnh được lưu theo định dạng ID.jpg
        img_path = image_folder_path / f"{img_id}.jpg" 
        
        if img_path.exists():
            caption = pred_list[0]['caption']
            # Load ảnh
            img = Image.open(img_path).convert("RGB")
            
            # Đẩy lên tensor
            score = clip_scorer(
                [img], 
                [caption]
            ).item()
            
            total_score += score
            valid_images += 1
            
    if valid_images == 0:
        return 0
        
    avg_clipscore = total_score / valid_images
    return avg_clipscore

# FLICKR8K_IMAGES_DIR = PROJECT_ROOT / "data" / "raw" / "Images"
# avg_cs = calculate_clipscore(res, FLICKR8K_IMAGES_DIR)
# print(f"\nAverage CLIPScore: {round(avg_cs, 4)}")